# Milestone 2 on Kaggle: 5 configs x 5 data fractions x 3 seeds

Settings: **Accelerator = GPU T4 x2** (only the first GPU is used), **Internet = on**. Run with *Save Version > Save & Run All*.

1. clones the repository and checks the frozen split with its tests;
2. downloads EuroSAT RGB from the authors' Zenodo record (MIT license) and checks its MD5;
3. smoke-tests the two ViT-S configs (tiny budget, results discarded) so a weight-loading problem shows up in the first minutes, not after hours;
4. copies the 9 milestone 1 runs from the repository into the output folder, so they are skipped rather than recomputed;
5. runs the 66 remaining runs, cheapest configs first, zipping the results after each config so a stopped session still leaves something to download;
6. builds the summary table.

All runs use a T4 so training times stay comparable with milestone 1.

In [ ]:
import os, shutil, subprocess, torch, timm
print('torch', torch.__version__, '| timm', timm.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the GPU accelerator')
print('CPU cores:', os.cpu_count(), '| free disk in /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9))

def run(cmd):
    """Run a shell command, stream its output, raise if it fails."""
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

In [ ]:
run('rm -rf /tmp/repo && git clone --depth 1 https://github.com/amine-belkacem/eurosat-onboard-transfer.git /tmp/repo')
os.chdir('/tmp/repo')
os.environ['PYTHONPATH'] = 'src'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
run('git log -1 --format="commit %h %s"')
run('python -m pytest -q tests/test_splits.py')

In [ ]:
run('python scripts/download_eurosat.py --data-dir /tmp/data')
DATA = '/tmp/data/EuroSAT_RGB/EuroSAT_RGB'

In [ ]:
for cfg in ['vit_small_probe', 'vit_small_ft']:
    run(f'python scripts/run_experiment.py --config configs/{cfg}.yaml --pcts 1 --seeds 0 --smoke '
        f'--data-root {DATA} --out-dir /tmp/smoke --overwrite')

In [ ]:
OUT = '/kaggle/working/results'
os.makedirs(OUT, exist_ok=True)
for cfg in ['cnn_scratch', 'resnet50_probe', 'resnet50_ft']:
    shutil.copytree(f'results/{cfg}', f'{OUT}/{cfg}', dirs_exist_ok=True)
run(f'ls {OUT}/*')

In [ ]:
CONFIGS = ['cnn_scratch', 'resnet50_probe', 'vit_small_probe', 'resnet50_ft', 'vit_small_ft']
for cfg in CONFIGS:
    run(f'python scripts/run_experiment.py --config configs/{cfg}.yaml --pcts 1 5 10 25 100 --seeds 0 1 2 '
        f'--data-root {DATA} --out-dir {OUT}')
    run('cd /kaggle/working && rm -f results.zip && zip -qr results.zip results')

In [ ]:
run(f'python scripts/summarize.py --results-dir {OUT}')
run('cd /kaggle/working && rm -f results.zip && zip -qr results.zip results && ls -lh results.zip')